# Retail portfolio strategy lab

Compare every automated monthly purchase strategy on real daily SPY data, inspect untouched holdout performance, then build one small strategy stack. Run the notebook top to bottom until the comparison charts appear; edit `APPROVED_STRATEGIES` before running the stacking section when manual review is preferred.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

import pandas as pd

from retail_sp500 import (
    LabConfig,
    build_stack,
    compare_strategies,
    comparison_figures,
    export_results,
    load_market,
    market_summary,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

## Configuration

In [ ]:
SYMBOL = "SPY"
START_DATE = "2007-06-01"
CACHE_PATH = Path("data/spy_daily.csv")
REFRESH = False
OUTPUT_DIR = Path("results/portfolio")

CONFIG = LabConfig(
    initial_cash=100_000,
    monthly_contribution=1_000,
    salary_day=1,
    holdout_years=4,
    stack_max_components=4,
)

## Load real daily market data

The first fetch requires `TWELVE_DATA_API_KEY`. Later runs use the ignored CSV cache unless `REFRESH = True`.

In [ ]:
daily = load_market(
    os.getenv("TWELVE_DATA_API_KEY"),
    cache_path=CACHE_PATH,
    refresh=REFRESH,
    symbol=SYMBOL,
    start_date=START_DATE,
)
market_summary(daily, symbol=SYMBOL)

## Run every strategy

The latest holdout years are not used to choose strategies or stack weights. `worth_testing` is based only on the earlier selection period.

In [ ]:
comparison = compare_strategies(daily, config=CONFIG)
print({
    "evaluation_start": comparison.evaluation_start.date().isoformat(),
    "holdout_start": comparison.holdout_start.date().isoformat(),
    "strategies": len(comparison.metrics),
})

In [ ]:
SELECTION_COLUMNS = [
    "key",
    "strategy",
    "family",
    "selection_calmar_ratio",
    "selection_annualized_return",
    "selection_max_drawdown",
    "natural_fill_rate",
    "forced_fill_rate",
    "worth_testing",
]
comparison.metrics[SELECTION_COLUMNS].head(30)

In [ ]:
pre_stack_figures = comparison_figures(comparison)
pre_stack_figures["selection_calmar"].show()
pre_stack_figures["selection_return_drawdown"].show()
pre_stack_figures["fill"].show()

## Approve strategies for stacking

Stop here and review only the selection-period table and charts above. Leave this as `None` to use the automatic pre-holdout filter, or enter strategy keys to enforce your own shortlist. The holdout remains unseen until the final section.

In [ ]:
APPROVED_STRATEGIES = None

# Example manual shortlist:
# APPROVED_STRATEGIES = [
#     "fixed-0.0050-10",
#     "fill-0.90-20",
# ]

## Build the stack, then reveal the untouched holdout

In [ ]:
stack = build_stack(
    comparison,
    config=CONFIG,
    approved_strategies=APPROVED_STRATEGIES,
)

stack.weights.rename("weight").to_frame()

In [ ]:
pd.Series(stack.metrics, name="selected_stack").to_frame()

In [ ]:
stack.selection_steps

In [ ]:
final_figures = comparison_figures(comparison, stack)
final_figures["stack_weights"].show()
final_figures["calmar"].show()
final_figures["return_drawdown"].show()
final_figures["wealth"].show()
final_figures["drawdown"].show()

## Export the comparison, stack, and interactive charts

In [ ]:
export_results(comparison, stack, output_dir=OUTPUT_DIR)